# Task 2.1 — Feature audit

This notebook audits the HDF5 data product produced by the final C++ extractor. It works on the 7k-row test file or the full ~1.4M-row file.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/python')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.feature_audit import *

In [2]:
# Set these paths before running.
H5_PATHS = [
    #'/work/clas12b/users/skuditha/ALERT/alert_pid/data/file1.h5',
    '/work/clas12b/users/skuditha/ALERT/alert_pid/data/full_dataset.h5',
]
LABEL_MAP_PATH = '/work/clas12b/users/skuditha/ALERT/alert_pid/config/label_map.json'
OUTDIR = Path('/work/clas12b/users/skuditha/ALERT/alert_pid/reports/feature_audit')

print('Edit H5_PATHS first.')

Edit H5_PATHS first.


In [3]:
# Load dataset
ds = load_audit_dataset(H5_PATHS, LABEL_MAP_PATH)
print({'n_rows': ds.n_rows, 'n_features': ds.n_features, 'files': [str(p) for p in ds.paths]})
print(ds.feature_names)

{'n_rows': 995087, 'n_features': 38, 'files': ['/work/clas12b/users/skuditha/ALERT/alert_pid/data/full_dataset.h5']}
['px', 'py', 'pz', 'p', 'pt', 'theta', 'phi', 'vx', 'vy', 'vz', 'vr', 'v3', 'n_hits', 'sum_adc', 'path', 'dEdx', 'dedx_recomputed', 'p_drift', 'sum_residuals', 'residual_per_hit', 'adc_per_hit', 'tof_time', 'pathlength', 'cluster_x', 'cluster_y', 'cluster_z', 'cluster_energy', 'n_bar', 'n_wedge', 'beta', 'm2', 'log_p', 'log_pt', 'log_sum_adc', 'log_path', 'log_dEdx', 'log_dedx_recomputed', 'log_cluster_energy']


In [4]:
class_balance = compute_class_balance(ds)
feature_summary = compute_feature_summary(ds)
mask_summary = compute_mask_summary(ds)
unit_sanity = infer_unit_sanity(ds)
pathologies = detect_pathologies(ds)
separation = compute_separation_table(ds)
pair_focus = pair_focus_summary(ds)

class_balance

,class_index,class_name,count,fraction
0,0,proton,144733,0.145448
1,1,deuteron,178294,0.179174
2,2,triton,190222,0.191161
3,3,helium3,243533,0.244735
4,4,helium4,238305,0.239482


In [5]:
feature_summary.head(20)

,feature,valid_count,invalid_count,valid_fraction,raw_min,raw_max,valid_min,valid_max,valid_mean,valid_std,zeros_in_stored_values
0,m2,969475,25612,0.974262,0.000000,1.137632e+09,2.059959,1.137632e+09,9.819962e+06,1.978162e+07,25612
1,adc_per_hit,995087,0,1.000000,33.500000,3.827000e+03,33.500000,3.827000e+03,8.470363e+02,6.234924e+02,0
2,beta,995087,0,1.000000,0.053991,1.199857e+00,0.053991,1.199857e+00,3.654518e-01,2.281131e-01,0
3,cluster_energy,995087,0,1.000000,0.393653,3.794149e+01,0.393653,3.794149e+01,8.097941e+00,6.210851e+00,0
4,cluster_x,995087,0,1.000000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,-1.055159e+00,6.295497e+01,0
5,cluster_y,995087,0,1.000000,-89.876656,8.987666e+01,-89.876656,8.987666e+01,2.055005e-01,6.210716e+01,0
6,cluster_z,995087,0,1.000000,-289.356232,2.895192e+02,-289.356232,2.895192e+02,-2.915092e+00,8.212947e+01,0
7,dEdx,995087,0,1.000000,0.435422,3.720885e+02,0.435422,3.720885e+02,7.950409e+01,6.462491e+01,0
8,dedx_recomputed,995087,0,1.000000,0.435422,3.720885e+02,0.435422,3.720885e+02,7.950409e+01,6.462491e+01,0
9,log_cluster_energy,995087,0,1.000000,-0.932286,3.636045e+00,-0.932286,3.636045e+00,1.683028e+00,1.027193e+00,0


In [6]:
mask_summary

,metric,value
0,rows_with_any_masked_feature,25612.000000
1,rows_with_no_masked_feature,969475.000000
2,mean_invalid_features_per_row,0.025738
3,max_invalid_features_in_row,1.000000


In [7]:
unit_sanity

,check,value,comment
0,p_median,591.626709,"Large O(1) suggests GeV/c, O(100-1000) suggest..."
1,tof_time_median_ns,1.313751,Expected ns-scale positive cluster timing.
2,pathlength_median,114.017540,Check whether pathlength looks mm-scale rather...
3,beta_median_stored,0.295310,Should be comfortably below 1 for most rows.
4,beta_median_recomputed_mm_ns,0.295310,Recomputed with c = 299.792458 mm/ns.
5,frac_beta_gt_1p0_stored,0.025738,Diagnostic only; no row cuts in audit.
6,frac_beta_gt_1p0_recomputed,0.025738,High value flags a unit mismatch or timing pat...
7,frac_m2_negative,0.000000,"Negative m2 can occur, but large fractions des..."


In [8]:
pathologies

,pathology,count
0,nonfinite_stored_values,0
1,rows_with_any_nonfinite_stored_value,0
2,valid_time_le_zero,0
3,valid_pathlength_le_zero,0
4,valid_p_le_zero,0
5,valid_dEdx_le_zero,0
6,valid_cluster_energy_le_zero,0
7,valid_beta_le_zero,0
8,valid_beta_gt_1p2,0


In [9]:
separation.head(15)

,feature,fisher_score
0,log_sum_adc,1.818748
1,log_dEdx,1.607374
2,log_dedx_recomputed,1.607374
3,adc_per_hit,1.583753
4,cluster_energy,1.461247
5,sum_adc,1.440211
6,log_cluster_energy,1.296753
7,dEdx,1.093219
8,dedx_recomputed,1.093219
9,residual_per_hit,0.130946


In [10]:
pair_focus

,feature,class_name,count,median,p16,p84
0,p,deuteron,178294,6.620948e+02,413.325714,1.179526e+03
1,p,helium4,238305,5.607089e+02,381.335540,9.309253e+02
2,tof_time,deuteron,178294,1.277626e+00,0.667122,2.091140e+00
3,tof_time,helium4,238305,1.412498e+00,0.769623,2.225502e+00
4,pathlength,deuteron,178294,1.140175e+02,91.082382,1.531912e+02
5,pathlength,helium4,238305,1.140175e+02,91.082382,1.548419e+02
6,dEdx,deuteron,178294,2.552408e+01,11.291241,5.979286e+01
7,dEdx,helium4,238305,1.290438e+02,82.808589,1.986913e+02
8,cluster_energy,deuteron,178294,3.149221e+00,1.108934,5.819423e+00
9,cluster_energy,helium4,238305,1.415954e+01,7.873053,1.930157e+01


In [11]:
corr = compute_correlation_matrix(ds)
high_corr = high_correlation_pairs(corr, threshold=0.95)
high_corr.head(30)

,feature_a,feature_b,corr
0,dEdx,dedx_recomputed,1.000000
1,log_dEdx,log_dedx_recomputed,1.000000
2,p,p_drift,0.999999
3,sum_residuals,residual_per_hit,0.990999
4,path,log_path,0.984949
5,sum_adc,adc_per_hit,0.982127
6,log_sum_adc,log_dedx_recomputed,0.954591
7,log_sum_adc,log_dEdx,0.954591


In [12]:
OUTDIR.mkdir(parents=True, exist_ok=True)
plot_feature_histograms(ds, KEY_PHYSICS_FEATURES, OUTDIR / 'key_histograms')
plot_feature_histograms(ds, DEFAULT_FEATURE_NAMES, OUTDIR / 'histograms')
plot_scatter_by_class(ds, 'p', 'beta', OUTDIR / 'beta_vs_p.png')
plot_scatter_by_class(ds, 'p', 'm2', OUTDIR / 'm2_vs_p.png')
plot_scatter_by_class(ds, 'p', 'dEdx', OUTDIR / 'dEdx_vs_p.png')
plot_scatter_by_class(ds, 'pathlength', 'cluster_energy', OUTDIR / 'cluster_energy_vs_pathlength.png')
plot_correlation_heatmap(corr, OUTDIR / 'correlation_heatmap.png')
print(f'Plots written under {OUTDIR.resolve()}')

Plots written under /ceph24/hallb/clas12/users/skuditha/ALERT/alert_pid/reports/feature_audit


In [ ]:
# One-shot batch run
# tables = run_full_feature_audit(H5_PATHS, LABEL_MAP_PATH, OUTDIR)